# TaskQueue Distribution Explorer

Interactive exploration of TaskQueue distributions extracted from a production DIRAC database.

**Setup:** Point `DATA_DIR` below to the directory containing the dated CSV files.

In [ ]:
from pathlib import Path
import glob

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 150, "figure.figsize": (12, 6), "font.size": 10})

COLORS = {
    "primary": "#2196F3",
    "secondary": "#FF9800",
    "accent": "#4CAF50",
    "dark": "#37474F",
}

DATA_DIR = Path("lhcb/data")


def load_csv(name: str) -> pd.DataFrame | None:
    """Load a CSV by base name, finding the latest dated version."""
    matches = sorted(DATA_DIR.glob(f"{name}_*.csv"))
    if not matches:
        print(f"  Skipping {name}: no matching CSV in {DATA_DIR}")
        return None
    path = matches[-1]  # latest date
    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        print(f"  Skipping {name}: empty file")
        return None
    if df.empty:
        print(f"  Skipping {name}: no data rows")
        return None
    print(f"  Loaded {path.name} ({len(df)} rows)")
    return df

## Scale Overview

In [ ]:
df = load_csv("scale_overview")
if df is not None:
    for col in df.columns:
        print(f"  {col}: {df[col].iloc[0]:,}")

## Site Distribution

In [ ]:
df = load_csv("site_distribution")
df_any = load_csv("site_any")

if df is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

    top = df.head(20).copy()
    top["site_short"] = top["site"].apply(lambda s: s if len(s) <= 25 else s[:22] + "...")
    ax1.barh(range(len(top)), top["total_jobs"], color=COLORS["primary"])
    ax1.set_yticks(range(len(top)))
    ax1.set_yticklabels(top["site_short"], fontsize=8)
    ax1.invert_yaxis()
    ax1.set_xlabel("Number of waiting jobs")
    ax1.set_title("Top 20 sites by waiting jobs")

    if df_any is not None and not df_any.empty:
        any_jobs = df_any["jobs_any_site"].iloc[0]
        ax1.axvline(x=any_jobs, color="red", linestyle="--", alpha=0.7)
        ax1.text(any_jobs, len(top) - 1, f'  "Any site": {any_jobs:,}',
                 color="red", fontsize=8, va="center")

    df_sorted = df.sort_values("total_jobs", ascending=False).reset_index(drop=True)
    df_sorted["cumulative_pct"] = df_sorted["total_jobs"].cumsum() / df_sorted["total_jobs"].sum() * 100
    ax2.plot(range(1, len(df_sorted) + 1), df_sorted["cumulative_pct"],
             color=COLORS["dark"], linewidth=2)
    ax2.axhline(y=80, color="red", linestyle="--", alpha=0.5, label="80%")
    ax2.axhline(y=95, color="orange", linestyle="--", alpha=0.5, label="95%")
    n_80 = (df_sorted["cumulative_pct"] <= 80).sum() + 1
    ax2.axvline(x=n_80, color="red", linestyle=":", alpha=0.3)
    ax2.text(n_80 + 1, 75, f"{n_80} sites = 80%", fontsize=9, color="red")
    ax2.set_xlabel("Number of sites (ranked by job count)")
    ax2.set_ylabel("Cumulative % of jobs")
    ax2.set_title("Site concentration (Pareto)")
    ax2.legend()

    fig.suptitle("Site Distribution", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

## Tag Distribution

In [ ]:
df = load_csv("tag_distribution")

if df is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

    top = df.head(20).copy()
    top["tag_short"] = top["tag"].apply(lambda s: s if len(s) <= 30 else s[:27] + "...")
    ax1.barh(range(len(top)), top["total_jobs"], color=COLORS["accent"])
    ax1.set_yticks(range(len(top)))
    ax1.set_yticklabels(top["tag_short"], fontsize=8)
    ax1.invert_yaxis()
    ax1.set_xlabel("Number of waiting jobs")
    ax1.set_title(f"Top 20 tags (of {len(df)} total)")

    tag_counts = df.groupby("n_tqs").size().reset_index(name="count")
    ax2.bar(tag_counts["n_tqs"].astype(str), tag_counts["count"], color=COLORS["secondary"])
    ax2.set_xlabel("Number of TQs using tag")
    ax2.set_ylabel("Number of tags")
    ax2.set_title("Tag popularity distribution")
    if len(tag_counts) > 15:
        ax2.tick_params(axis="x", rotation=45)

    fig.suptitle("Tag Distribution", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

## CPU Time Distribution

In [ ]:
df = load_csv("cputime_distribution")

if df is not None:
    def format_time(seconds):
        if seconds < 3600:
            return f"{seconds // 60}m"
        elif seconds < 86400:
            return f"{seconds // 3600}h"
        else:
            d = seconds / 86400
            return f"{d:.1f}d" if d != int(d) else f"{int(d)}d"

    df["label"] = df["cpu_segment_seconds"].apply(format_time)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    ax1.bar(range(len(df)), df["total_jobs"], color=COLORS["primary"])
    ax1.set_xticks(range(len(df)))
    ax1.set_xticklabels(df["label"], rotation=45, ha="right")
    ax1.set_ylabel("Number of waiting jobs")
    ax1.set_title("Jobs by CPU time segment")

    ax2.bar(range(len(df)), df["n_tqs"], color=COLORS["secondary"])
    ax2.set_xticks(range(len(df)))
    ax2.set_xticklabels(df["label"], rotation=45, ha="right")
    ax2.set_ylabel("Number of TaskQueues")
    ax2.set_title("TaskQueues by CPU time segment")

    fig.suptitle("CPU Time Distribution", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

## RAM Distribution

In [ ]:
df = load_csv("ram_distribution")
df_none = load_csv("ram_none")

if df is not None or df_none is not None:
    fig, ax = plt.subplots(figsize=(12, 6))

    if df is not None and not df.empty:
        df["label"] = df.apply(lambda r: f"{r['MinRAM']}-{r['MaxRAM']} MB", axis=1)
        ax.bar(range(len(df)), df["total_jobs"], color=COLORS["primary"], label="With RAM req")
        ax.set_xticks(range(len(df)))
        ax.set_xticklabels(df["label"], rotation=45, ha="right", fontsize=8)

    if df_none is not None and not df_none.empty:
        no_ram_jobs = df_none["jobs_no_ram"].iloc[0]
        no_ram_tqs = df_none["tqs_no_ram"].iloc[0]
        ax.text(0.95, 0.95, f"No RAM requirement:\n{no_ram_tqs:,} TQs, {no_ram_jobs:,} jobs",
                transform=ax.transAxes, ha="right", va="top",
                bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8), fontsize=9)

    ax.set_ylabel("Number of waiting jobs")
    ax.set_title("RAM Requirements Distribution", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

## Job Type Distribution

In [ ]:
df = load_csv("jobtype_distribution")

if df is not None:
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.barh(range(len(df)), df["total_jobs"], color=COLORS["primary"])
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df["job_type"], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Number of waiting jobs")

    for i, row in df.iterrows():
        ax.text(row["total_jobs"], i, f"  ({row['n_tqs']} TQs)", va="center", fontsize=8, color="gray")

    ax.set_title("Job Type Distribution", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

## TaskQueue Size Distribution

In [ ]:
df = load_csv("tq_size_histogram")

if df is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    ax1.bar(range(len(df)), df["n_tqs"], color=COLORS["secondary"])
    ax1.set_xticks(range(len(df)))
    ax1.set_xticklabels(df["size_bucket"], rotation=30)
    ax1.set_ylabel("Number of TaskQueues")
    ax1.set_title("TaskQueues by size bucket")

    ax2.bar(range(len(df)), df["total_jobs"], color=COLORS["primary"])
    ax2.set_xticks(range(len(df)))
    ax2.set_xticklabels(df["size_bucket"], rotation=30)
    ax2.set_ylabel("Total jobs in bucket")
    ax2.set_title("Jobs by TQ size bucket")

    fig.suptitle("TaskQueue Size Distribution", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

## Owner Distribution

In [ ]:
df = load_csv("owner_distribution")

if df is not None:
    fig, ax = plt.subplots(figsize=(14, 7))

    df["label"] = df["Owner"].str[:15] + " / " + df["OwnerGroup"]
    top = df.head(20)

    ax.barh(range(len(top)), top["total_jobs"], color=COLORS["primary"])
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top["label"], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Number of waiting jobs")
    ax.set_title("Top 20 Owners by Waiting Jobs", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

## Platform Distribution

In [ ]:
df = load_csv("platform_distribution")

if df is not None:
    fig, ax = plt.subplots(figsize=(10, 5))

    ax.bar(range(len(df)), df["total_jobs"], color=COLORS["accent"])
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df["platform"], rotation=30, ha="right")
    ax.set_ylabel("Number of waiting jobs")
    ax.set_title("Platform Distribution", fontsize=14, fontweight="bold")

    for i, row in df.iterrows():
        ax.text(i, row["total_jobs"], f"{row['n_tqs']} TQs", ha="center", va="bottom", fontsize=8, color="gray")

    fig.tight_layout()
    plt.show()

## Running Jobs by Site

In [ ]:
df = load_csv("running_by_site")

if df is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

    top = df.head(20).copy()
    top["site_short"] = top["Site"].apply(lambda s: s if len(s) <= 25 else s[:22] + "...")

    ax1.barh(range(len(top)), top["n_running"], color=COLORS["dark"])
    ax1.set_yticks(range(len(top)))
    ax1.set_yticklabels(top["site_short"], fontsize=8)
    ax1.invert_yaxis()
    ax1.set_xlabel("Number of running jobs")
    ax1.set_title("Top 20 sites by running jobs")

    df_sorted = df.sort_values("n_running", ascending=False).reset_index(drop=True)
    df_sorted["cumulative_pct"] = df_sorted["n_running"].cumsum() / df_sorted["n_running"].sum() * 100
    ax2.plot(range(1, len(df_sorted) + 1), df_sorted["cumulative_pct"],
             color=COLORS["dark"], linewidth=2)
    ax2.axhline(y=80, color="red", linestyle="--", alpha=0.5, label="80%")
    ax2.axhline(y=95, color="orange", linestyle="--", alpha=0.5, label="95%")
    ax2.set_xlabel("Number of sites (ranked)")
    ax2.set_ylabel("Cumulative % of running jobs")
    ax2.set_title("Running jobs concentration")
    ax2.legend()

    fig.suptitle("Running Jobs Distribution (pilot-side view)", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()